In [1]:
import gifba
from scipy.integrate import solve_ivp
import numpy as np
import cobra as cb
import pandas as pd
from IPython.display import clear_output
import copy
import matplotlib.pyplot as plt
from cobra import Model
import os
import time
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm

def rename_community_biomass(models, old_biomass_id="EX_Bio(e)"):
    """
    Renames the biomass exchange reactions and their associated metabolites 
    to track individual species separately.
    """
    # Deepcopy ensures we don't accidentally modify the original template models
    updated_models = copy.deepcopy(models)
    
    for idx, model in enumerate(updated_models):
        suffix = f"_{idx + 1}"  # e.g., _1, _2
        
        if old_biomass_id in model.reactions:
            # 1. Isolate the biomass reaction
            biomass_rxn = model.reactions.get_by_id(old_biomass_id)
            
            # 2. Rename the associated biomass metabolite(s) FIRST
            # We cast to list() to avoid modifying a dictionary while iterating over it
            for metabolite in list(biomass_rxn.metabolites):
                metabolite.id = f"{metabolite.id}{suffix}"
                # Optional: update the metabolite name for clarity
                metabolite.name = f"{metabolite.name} (Species {idx + 1})"
            
            # 3. Rename the reaction itself
            biomass_rxn.id = f"{old_biomass_id}{suffix}"
            biomass_rxn.name = f"{biomass_rxn.name} (Species {idx + 1})"
            
    return updated_models

# n_dfba_fba_calls = 0  # Global counter for FBA calls during DFBA
def dsdt(t, S, models, vmax, km, media_keys, rel_abund, record_fluxes=None):
    global n_dfba_fba_calls
    dsdt_rates = np.zeros(len(S))
    conc_map = dict(zip(media_keys, S))
    
    # Collect fluxes for logging if a list is provided
    model_flux_records = {} if record_fluxes is not None else None
    
    for mdl_idx, model in enumerate(models):
        for ex_id in media_keys:
            if ex_id in model.reactions and "bio" not in ex_id:
                v_max = vmax.loc[str(mdl_idx), ex_id]
                k_m = km.loc[str(mdl_idx), ex_id]
                c = conc_map[ex_id]
                
                if c < 1e-6:
                    c = 0.0

                uptake_limit = (v_max * c) / (k_m + c)
                model.reactions.get_by_id(ex_id).lower_bound = -uptake_limit

        # use pfba to get the optimal flux distribution
        solution = cb.flux_analysis.parsimonious.pfba(model)
        # solution = model.optimize()
        n_dfba_fba_calls += 1  # Increment the global FBA call counter

        if solution.status == 'optimal':
            # Save fluxes for this specific model if recording
            if model_flux_records is not None:
                # scale fluxes by relative abundance
                scaled_fluxes = {rxn: flux * rel_abund[mdl_idx] for rxn, flux in solution.fluxes.items()}
                model_flux_records[mdl_idx] = scaled_fluxes
                
            for i, ex_id in enumerate(media_keys):
                flux_per_hr = solution.fluxes.get(ex_id, 0)
                dsdt_rates[i] += flux_per_hr * rel_abund[mdl_idx]
        else:
            if model_flux_records is not None:
                model_flux_records[mdl_idx] = {}

    for i, ex_id in enumerate(media_keys):
        media_flux = community.env_fluxes.loc[0, ex_id]
        dsdt_rates[i] += media_flux 

    if record_fluxes is not None and model_flux_records:
        record_fluxes.append((t, model_flux_records))

    return dsdt_rates


class dFBATracker:
    def __init__(self, t_start, t_end, tol=1e-5):
        self.pbar = tqdm(total=t_end - t_start, desc="Simulating ODE")
        self.highest_t = t_start
        self.tol = tol
        
        self.prev_t = None
        self.prev_dsdt = None
        self.curr_t = None
        self.curr_dsdt = None
        
        # Storage for time-series flux tracking
        self.flux_history = []

    def ode_wrapper(self, t, S, models, vmax, km, media_keys, rel_abund):
        # Pass self.flux_history to capture fluxes during solve_ivp evaluations
        rates = dsdt(t, S, models, vmax, km, media_keys, rel_abund, record_fluxes=self.flux_history)
        
        if t > self.highest_t:
            self.pbar.update(t - self.highest_t)
            self.highest_t = t
        
        if self.curr_t is None or abs(t - self.curr_t) > 1e-4:
            self.prev_t = self.curr_t
            self.prev_dsdt = self.curr_dsdt
            
        self.curr_t = t
        self.curr_dsdt = rates
        
        return rates

    def check_stop_condition(self, t, S, models, vmax, km, media_keys, rel_abund):
        # 1. Get current rates (do NOT pass record_fluxes here so we don't pollute the logs)
        current_rates = dsdt(t, S, models, vmax, km, media_keys, rel_abund)
        
        # 2. Take a small, controlled mathematical step forward (finite difference)
        eps = 1e-4
        S_next = S + (current_rates * eps)
        
        # 3. Get the rates at this simulated future step
        future_rates = dsdt(t + eps, S_next, models, vmax, km, media_keys, rel_abund)
        
        # 4. Calculate maximum acceleration (change in rates / time)
        max_accel = np.max(np.abs(future_rates - current_rates)) / eps
        
        # When max_accel drops below self.tol, this crosses 0 and stops the solver
        return max_accel - self.tol

    def get_flux_dataframe(self, models, sol, decimals=6):
        """Builds the DataFrame matching only accepted solver timestamps in sol.t."""
        # Round sol.t timestamps for matching
        accepted_times = set(np.round(sol.t, decimals))
        
        rows = []
        index_tuples = []
        all_reactions = sorted(list(set(rxn.id for model in models for rxn in model.reactions)))
        
        for t, model_dict in self.flux_history:
            t_rounded = round(t, decimals)
            # Only keep evaluations that correspond to accepted ODE solver steps
            if t_rounded in accepted_times:
                for mdl_idx in range(len(models)):
                    fluxes = model_dict.get(mdl_idx, {})
                    index_tuples.append((t_rounded, mdl_idx))
                    row_data = {rxn: fluxes.get(rxn, 0.0) for rxn in all_reactions}
                    rows.append(row_data)
        multi_index = pd.MultiIndex.from_tuples(index_tuples, names=['Time', 'Model'])
        df_fluxes = pd.DataFrame(rows, index=multi_index, columns=all_reactions)
        
        # Clean up any residual duplicates from event root-finding
        df_deduped = df_fluxes[~df_fluxes.index.duplicated(keep='last')]
        return df_deduped.sort_index()

# run dFBA on toy models

In [ ]:
# load models & media for communities in main toy model fig
sims = {
    "1_3_parallel" : [0.5, 0.5],
    "2_1_crossfeed": [0.5, 0.5],
    "3_competition": [0.2, 0.8],
    "4_2_coupling" : [0.2, 0.8]
}

# kinetic params
V_MAX_ALL = 100  # (mmol/(gi*hr)) Maximum uptake rate for all substrates
K_M_ALL = 0.1  # (mmol) Michaelis-Menten constant for all substrates

sims_keys = list(sims.keys())
for sim_key in sims_keys:
    models, media = gifba.utils.load_simple_models(sim_key)

    ids_list = ["org1", "org2"]
    rel_abund = sims[sim_key]

    # rename biomass rxns per model
    renamed_models = rename_community_biomass(models, old_biomass_id="EX_Bio(e)")

    community = gifba.gifbaObject(renamed_models, media, rel_abund=rel_abund)

    start_time = time.time()
    env_flux, org_flux = community.run_gifba(n_iterations=6, method="pfba", verbose=False)
    end_time = time.time()
    print(f"periodicity: {community.periodicity}")
    print(f"giFBA run time: {end_time - start_time:.2f} seconds")

    # equal rel_abund
    rel_abund = community.rel_abund.ravel()

    # media from gifba
    substrate_0 = community.env_fluxes.loc[0, :].to_dict()


    vmax_vals = pd.DataFrame(np.full((len(renamed_models), len(substrate_0.keys())), V_MAX_ALL),
                                columns=list(substrate_0.keys()),
                                index=[str(mdl_idx) for mdl_idx in range(len(renamed_models))])
    km_vals = pd.DataFrame(np.full((len(renamed_models), len(substrate_0.keys())), K_M_ALL),
                                columns=list(substrate_0.keys()),
                                index=[str(mdl_idx) for mdl_idx in range(len(renamed_models))])

    # Preparing cleaner inputs for solve_ivp
    m_keys = list(substrate_0.keys()) 
    m_vals = np.array([substrate_0[k] for k in m_keys]) 
    y0 = np.zeros(len(m_vals))  

    # solve_ivp time
    t_start, t_end = 0, 100
    t_span = (t_start, t_end)

    # Initialize the tracker
    tracker = dFBATracker(t_start=t_start, t_end=t_end)

    n_dfba_fba_calls = 0  # Reset the global FBA call counter before running solve_ivp
    time_start = time.time()
    sol = solve_ivp(
        fun=tracker.ode_wrapper,
        t_span=t_span,
        y0=y0,
        args=(renamed_models, vmax_vals, km_vals, m_keys, rel_abund),
        # events=tracker.check_stop_condition,
        method='BDF'
    )
    time_end = time.time()
    # Retrieve the 3D-equivalent multi-indexed DataFrame
    df_flux_timeseries = tracker.get_flux_dataframe(renamed_models, sol)


    # biomass concentrations from the ODE solution; their time derivative is the growth flux
    bio1_conc = sol.y[m_keys.index("EX_Bio(e)_1")]
    bio2_conc = sol.y[m_keys.index("EX_Bio(e)_2")]
    org1_bio_flux_dfba = np.gradient(bio1_conc, sol.t)
    org2_bio_flux_dfba = np.gradient(bio2_conc, sol.t)

    # giFBA converged biomass flux of each organism (org_flux is indexed by model)
    org1_bio_flux_gifba = org_flux.loc[0, "EX_Bio(e)_1"]
    org2_bio_flux_gifba = org_flux.loc[1, "EX_Bio(e)_2"]

    plt.figure(figsize=(7, 4), clear=True)

    if np.allclose(org1_bio_flux_dfba, org2_bio_flux_dfba, atol=1e-3):
        plt.plot(sol.t, org1_bio_flux_dfba, label="Org. 1 & 2 dFBA", lw=2, color=plt.cm.tab10(3))
    else:
        plt.plot(sol.t, org1_bio_flux_dfba, label="Org. 1 dFBA", lw=2, color=plt.cm.tab20(6), zorder=2)
        plt.plot(sol.t, org2_bio_flux_dfba, label="Org. 2 dFBA", lw=4, color=plt.cm.tab20(7), zorder=1)

    if np.isclose(org1_bio_flux_gifba, org2_bio_flux_gifba, atol=1e-3):
        ls = "--" if sim_key != "4_2_coupling" else (5, (10, 3))
        plt.axhline(org1_bio_flux_gifba, label="Org. 1 & 2 giFBA Sol.", ls=ls, color=plt.cm.tab10(0))
    else:
        plt.axhline(org1_bio_flux_gifba, label="Org. 1 giFBA Sol.", ls="--", color=plt.cm.tab20(0), zorder=3)
        plt.axhline(org2_bio_flux_gifba, label="Org. 2 giFBA Sol.", ls="--", lw=2, color=plt.cm.tab20(1), zorder=3)

    # hard coded period 2 oscillations for the 4_3_coupling case (named 4_2 for loading models)
    if sim_key == "4_2_coupling":
        # plot period 2 points for giFBA
        period_2_p1 = community.org_fluxes.loc[(0, community.n_iterations-community.periodicity), "EX_Bio(e)_1"] # both orgs have the period 2 fp, but phase shifted by 1 iteration
        period_2_p2 = community.org_fluxes.loc[(0, community.n_iterations-community.periodicity+1), "EX_Bio(e)_1"]   

        # type 1 - bars
        plt.axhline(period_2_p1, ls=(2, (4, 2, 1, 2)), lw=2, color="k", alpha= 0.5, label=r"giFBA Period-2 $p_1$", zorder=4)
        plt.axhline(period_2_p2, ls=(2, (1, 1)), lw=2, color="k", alpha= 0.5, label=r"giFBA Period-2 $p_2$", zorder=4)

        # type 2 - dots
        # period_times = 10.0 ** np.array(list(range(-5, 3)))
        # period_times = np.concatenate(([0], period_times)) # add zero
        # n_times = len(period_times)
        # period2_p1_half_array = np.full(n_times//2, period_2_p1).reshape(-1,1)
        # period2_p2_half_array = np.full(n_times//2, period_2_p2).reshape(-1,1)
        # period_2_fps_org1 = np.hstack((period2_p1_half_array, period2_p2_half_array)).flatten()
        # period_2_fps_org2 = np.hstack((period2_p2_half_array, period2_p1_half_array)).flatten()
        # period_2_fps_org1 = np.concatenate((period_2_fps_org1.flatten(), np.array([period_2_p1])))
        # period_2_fps_org2 = np.concatenate((period_2_fps_org2.flatten(), np.array([period_2_p2])))

        # plt.scatter(period_times, period_2_fps_org1, color=plt.cm.tab20(0), label=r"giFBA Period-2 (Org. 1)", zorder=4)
        # plt.scatter(period_times, period_2_fps_org2, color=plt.cm.tab20(1), label=r"giFBA Period-2 (Org. 2)", zorder=4)
        
    # anchor the legend's left edge just outside the axes; bbox_inches="tight" in savefig keeps it in frame
    plt.legend(fontsize=11, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, labelspacing=1.5)
    
    plt.xscale("symlog", linthresh=1e-5)
    plt.xlim(left=0, right=sol.t[-1])
    plt.xlabel("Time (hr)")
    plt.ylabel(r"Biomass flux $\left (\frac{g_i}{g_T \cdot hr} \right )$")
    plt.grid(True, which='both', ls='--', lw=0.5)
    plt.tight_layout()
    if sim_key == "3_competition":
        sim_key = "3_2_competition"  # Rename for consistency with other parts of the code
    elif sim_key == "4_2_coupling":
        sim_key = "4_3_coupling"  # Rename for consistency with other parts of the code
    plt.savefig(f"../figures/toy_model_figs/growth_growth_plots/dfba_vs_gifba_{sim_key}.png", dpi=600, bbox_inches="tight")
    plt.close()  # Close the figure to free memory


Read LP format model from file /tmp/tmprawrpn5o.lp
Reading time = 0.00 seconds
: 4 rows, 10 columns, 16 nonzeros
Read LP format model from file /tmp/tmp0vyjvmmq.lp
Reading time = 0.00 seconds
: 4 rows, 10 columns, 16 nonzeros
Read LP format model from file /tmp/tmpn025z9op.lp
Reading time = 0.00 seconds
: 4 rows, 10 columns, 16 nonzeros
Read LP format model from file /tmp/tmpjl901h17.lp
Reading time = 0.00 seconds
: 4 rows, 10 columns, 16 nonzeros


periodicity: 1
giFBA run time: 0.03 seconds


Simulating ODE: 100%|██████████| 100.0/100 [00:14<00:00,  6.71it/s]              


Read LP format model from file /tmp/tmpg7s9n99e.lp
Reading time = 0.00 seconds
: 6 rows, 16 columns, 26 nonzeros
Read LP format model from file /tmp/tmpzk8s28jr.lp
Reading time = 0.00 seconds
: 6 rows, 14 columns, 24 nonzeros
Read LP format model from file /tmp/tmpfcffiho4.lp
Reading time = 0.00 seconds
: 6 rows, 16 columns, 26 nonzeros
Read LP format model from file /tmp/tmphr822tuh.lp
Reading time = 0.00 seconds
: 6 rows, 14 columns, 24 nonzeros
periodicity: 1
giFBA run time: 0.04 seconds


Simulating ODE: 100%|██████████| 100.0/100 [00:00<00:00, 113.68it/s]             
Simulating ODE:   0%|          | 0.02898656371548783/100 [00:00<10:13,  6.14s/it] 

Read LP format model from file /tmp/tmp0ehjy6xc.lp
Reading time = 0.00 seconds
: 4 rows, 10 columns, 16 nonzeros
Read LP format model from file /tmp/tmpx8mrri7e.lp
Reading time = 0.00 seconds
: 4 rows, 10 columns, 16 nonzeros
Read LP format model from file /tmp/tmp6evftk39.lp
Reading time = 0.00 seconds
: 4 rows, 10 columns, 16 nonzeros
Read LP format model from file /tmp/tmp1vet05qt.lp
Reading time = 0.00 seconds
: 4 rows, 10 columns, 16 nonzeros
periodicity: 1
giFBA run time: 0.06 seconds


Simulating ODE: 100%|██████████| 100.0/100 [00:00<00:00, 103.85it/s]             


Read LP format model from file /tmp/tmpjakd3az8.lp
Reading time = 0.00 seconds
: 6 rows, 14 columns, 24 nonzeros
Read LP format model from file /tmp/tmpylvcqmfs.lp
Reading time = 0.00 seconds
: 6 rows, 14 columns, 24 nonzeros
Read LP format model from file /tmp/tmpzn5p7r6s.lp
Reading time = 0.00 seconds
: 6 rows, 14 columns, 24 nonzeros
Read LP format model from file /tmp/tmpwdbu6nz6.lp
Reading time = 0.00 seconds
: 6 rows, 14 columns, 24 nonzeros
periodicity: 2
giFBA run time: 0.06 seconds


Simulating ODE: 100%|██████████| 100.0/100 [00:00<00:00, 106.69it/s]            
Simulating ODE:   0%|          | 0.048147589346678094/100 [00:00<06:04,  3.65s/it]